In [31]:
# =========================
# 1. IMPORTS
# =========================
import numpy as np
import pandas as pd
import os
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2   # can change to ResNet50
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

In [32]:
# =========================
# 2. PATHS (CHANGE THIS ⚠️)
# =========================

# 👉 CHANGE according to dataset
TRAIN_DIR = "/kaggle/input/competitions/cs-460-muffin-vs-chihuahua-classification-challenge/train"
TEST_DIR  = "/kaggle/input/competitions/cs-460-muffin-vs-chihuahua-classification-challenge/kaggle_test_final"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

In [33]:
# =========================
# 3. DATA GENERATOR (OVERFITTING CONTROL 🔥)
# =========================

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,   # automatically creates validation set
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

train_data = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',  # 👉 change to 'categorical' if >2 classes
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

Found 3788 images belonging to 2 classes.
Found 945 images belonging to 2 classes.


In [34]:
# =========================
# 4. MODEL (TRANSFER LEARNING 🔥)
# =========================

base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

# Freeze base model (IMPORTANT for overfitting)
for layer in base_model.layers:
    layer.trainable = False

# Custom layers
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)

# 👉 OUTPUT LAYER CHANGE HERE ⚠️
output = Dense(1, activation='sigmoid')(x)  # binary

model = Model(inputs=base_model.input, outputs=output)


In [35]:

# =========================
# 5. COMPILE
# =========================

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',   # 👉 categorical_crossentropy if multi-class
    metrics=['accuracy']
)

In [36]:
# =========================
# 6. CALLBACKS (OVERFITTING CONTROL 🔥)
# =========================

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)


In [37]:

# =========================
# 7. TRAIN
# =========================

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5,   # 👉 keep low (5–10)
    callbacks=[early_stop]
)


Epoch 1/5
119/119 ━━━━━━━━━━━━━━━━━━━━ 84s 637ms/step - accuracy: 0.9463 - loss: 0.1180 - val_accuracy: 0.9915 - val_loss: 0.0279
Epoch 2/5
119/119 ━━━━━━━━━━━━━━━━━━━━ 64s 540ms/step - accuracy: 0.9866 - loss: 0.0395 - val_accuracy: 0.9873 - val_loss: 0.0374
Epoch 3/5
119/119 ━━━━━━━━━━━━━━━━━━━━ 64s 536ms/step - accuracy: 0.9938 - loss: 0.0186 - val_accuracy: 0.9894 - val_loss: 0.0310


In [38]:
# =========================
# 8. FINE-TUNING (EXTRA BOOST 🔥)
# =========================

# Unfreeze last few layers
for layer in base_model.layers[-20:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # low LR
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

Epoch 1/5
119/119 ━━━━━━━━━━━━━━━━━━━━ 90s 647ms/step - accuracy: 0.9249 - loss: 0.2183 - val_accuracy: 0.9905 - val_loss: 0.0338
Epoch 2/5
119/119 ━━━━━━━━━━━━━━━━━━━━ 66s 552ms/step - accuracy: 0.9758 - loss: 0.0604 - val_accuracy: 0.9926 - val_loss: 0.0289
Epoch 3/5
119/119 ━━━━━━━━━━━━━━━━━━━━ 66s 552ms/step - accuracy: 0.9844 - loss: 0.0436 - val_accuracy: 0.9915 - val_loss: 0.0242
Epoch 4/5
119/119 ━━━━━━━━━━━━━━━━━━━━ 64s 539ms/step - accuracy: 0.9842 - loss: 0.0436 - val_accuracy: 0.9926 - val_loss: 0.0279
Epoch 5/5
119/119 ━━━━━━━━━━━━━━━━━━━━ 64s 536ms/step - accuracy: 0.9899 - loss: 0.0305 - val_accuracy: 0.9915 - val_loss: 0.0323


In [39]:
test_images = [img for img in os.listdir(TEST_DIR) if img.endswith(('.jpg', '.png', '.jpeg'))]

df_test = pd.DataFrame({
    'filename': test_images
})

test_data = ImageDataGenerator(rescale=1./255).flow_from_dataframe(
    df_test,
    directory=TEST_DIR,
    x_col='filename',
    y_col=None,
    target_size=IMG_SIZE,
    class_mode=None,
    batch_size=1,
    shuffle=False
)

Found 1138 validated image filenames.


In [40]:
print(os.listdir(TEST_DIR))

['img_4_684.jpg', 'img_3_835.jpg', 'img_0_340.jpg', 'img_2_138.jpg', 'img_3_570.jpg', 'img_1_210.jpg', 'img_2_792.jpg', 'img_1_581.jpg', 'img_2_443.jpg', 'img_1_920.jpg', 'img_1_547.jpg', 'img_2_819.jpg', 'img_2_105.jpg', 'img_2_821.jpg', 'img_0_591.jpg', 'img_3_439.jpg', 'img_4_1.jpg', 'img_3_9.jpg', 'img_3_239.jpg', 'img_4_204.jpg', 'img_0_1180.jpg', 'img_4_776.jpg', 'img_2_85.jpg', 'img_1_1008.jpg', 'img_2_508.jpg', 'img_4_545.jpg', 'img_2_278.jpg', 'img_3_914.jpg', 'img_2_857.jpg', 'img_2_1017.jpg', 'img_2_763.jpg', 'img_1_174.jpg', 'img_4_882.jpg', 'img_1_909.jpg', 'img_1_754.jpg', 'img_3_275.jpg', 'img_1_526.jpg', 'img_4_69.jpg', 'img_4_929.jpg', 'img_3_839.jpg', 'img_2_681.jpg', 'img_1_1206.jpg', 'img_1_890.jpg', 'img_1_1192.jpg', 'img_4_1155.jpg', 'img_3_99.jpg', 'img_2_655.jpg', 'img_3_34.jpg', 'img_4_185.jpg', 'img_4_178.jpg', 'img_0_1169.jpg', 'img_1_343.jpg', 'img_0_105.jpg', 'img_4_1055.jpg', 'img_0_757.jpg', 'img_2_1043.jpg', 'img_0_833.jpg', 'img_1_287.jpg', 'img_1_22.jp

In [41]:
# =========================
# 10. PREDICTION
# =========================

preds = model.predict(test_data)

# 👉 THRESHOLD CHANGE IF NEEDED
preds = (preds > 0.5).astype(int)

1138/1138 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step


In [42]:
# =========================
# 11. SUBMISSION FILE
# =========================

filenames = test_data.filenames

submission = pd.DataFrame({
    "id": filenames,
    "label": preds.flatten()
})

submission.to_csv("submission.csv", index=False)

print("✅ Submission file created!")

✅ Submission file created!
